VIII.) Protein-coding sensitivity GO analysis for the batch-adjusted early Human-Chimpanzee PFC comparison

This notebook repeats the pathway analysis after restricting the tested gene universe to genes classified as protein-coding.

It contains four analyses:

1. Ranked directional GOfuncR as the reference analysis and the analysis criticized for using a gene-level randomization null.
2. Mean-t Freedman-Lane using moderated species t-statistics. This is the main sample-level sensitivity analysis because the corresponding full-universe test supported the checkpoint/mitotic module.
3. Mean-rank Freedman-Lane using the same moderated t-statistics. This is the rank-based sample-level version and was negative after multiple-testing correction in the full-universe analysis.
4. Rank-based CAMERA with estimated inter-gene correlation as the primary specification and fixed correlation 0.05 as a stress test. A fixed 0.01 run is available only as an optional reference and is disabled by default.

The same nine early Human and Chimpanzee samples and the same age + species + batch model are used throughout.

Positive species effects indicate Human-higher expression. Negative species effects indicate Chimpanzee-higher expression.

The 225 selected protein-coding DE genes are loaded and reported for context, but all ranked and competitive pathway tests use the entire protein-coding tested universe.


1. Packages


In [1]:
options(stringsAsFactors = FALSE)

suppressPackageStartupMessages({
    library(GOfuncR)
    library(Homo.sapiens)
    library(limma)
    library(AnnotationDbi)
    library(org.Hs.eg.db)
    library(GO.db)
    library(Matrix)
})

dir.create("results", showWarnings = FALSE, recursive = TRUE)

cat("Packages loaded.\n")


Packages loaded.


2. Settings

GOfuncR uses 10,000 random sets.

Both Freedman-Lane statistics are computed in the same 10,000 sample-level residual permutations with seed 42.

GO Biological Process sets containing 10 to 500 tested protein-coding genes are retained for Freedman-Lane and CAMERA.

BH FDR 0.10 and FWER 0.05 are used as before.

The fixed-correlation CAMERA run with correlation 0.01 is disabled by default because it was much more permissive in the full-universe analysis. Set RUN_CAMERA_001_REFERENCE to TRUE only if the reference comparison is desired.


In [ ]:
DE_RESULTS_FILE <- "results/early_human_chimp_pylimma_results_human_spec_batch_adjusted.csv"
EXPRESSION_FILE <- "results/pfc_mrna_gene_expression_human_spec.csv"
SAMPLE_FILE <- "results/early_human_chimp_pylimma_sample_table_human_spec_with_batch.csv"

PROTEIN_CODING_UNIVERSE_FILE <- "results/protein_coding_genes_full_tested_universe.csv"
PROTEIN_CODING_SELECTED_FILE <- "results/protein_coding_genes_selected_251.csv"

N_RANDSETS <- 10000L
N_PERMUTATIONS <- 10000L
RANDOM_SEED <- 42L

FWER_CUTOFF <- 0.05
FDR_CUTOFF <- 0.10
MIN_GO_SIZE <- 10L
MAX_GO_SIZE <- 500L
PERM_BLOCK <- 50L

RUN_CAMERA_001_REFERENCE <- TRUE

GOFUNCR_ALL_FILE <- "results/gofuncr_ranked_directional_protein_coding_10000.csv"
GOFUNCR_SIG_FILE <- "results/gofuncr_ranked_directional_protein_coding_FWER_005.csv"
GOFUNCR_REFINED_HUMAN_FILE <- "results/gofuncr_ranked_directional_protein_coding_refined_human.csv"
GOFUNCR_REFINED_CHIMP_FILE <- "results/gofuncr_ranked_directional_protein_coding_refined_chimp.csv"

FL_T_ALL_FILE <- "results/freedman_lane_meant_GO_BP_protein_coding_moderated_10000.csv"
FL_T_FDR_FILE <- "results/freedman_lane_meant_GO_BP_protein_coding_moderated_FDR_010.csv"
FL_T_FWER_FILE <- "results/freedman_lane_meant_GO_BP_protein_coding_moderated_FWER_005.csv"

FL_RANK_ALL_FILE <- "results/freedman_lane_meanrank_GO_BP_protein_coding_moderated_10000.csv"
FL_RANK_FDR_FILE <- "results/freedman_lane_meanrank_GO_BP_protein_coding_moderated_FDR_010.csv"
FL_RANK_FWER_FILE <- "results/freedman_lane_meanrank_GO_BP_protein_coding_moderated_FWER_005.csv"

CAM_EST_ALL_FILE <- "results/camera_rank_estcorr_GO_BP_protein_coding.csv"
CAM_EST_FDR_FILE <- "results/camera_rank_estcorr_GO_BP_protein_coding_FDR_010.csv"

CAM_005_ALL_FILE <- "results/camera_rank_fixed005_GO_BP_protein_coding.csv"
CAM_005_FDR_FILE <- "results/camera_rank_fixed005_GO_BP_protein_coding_FDR_010.csv"

CAM_001_ALL_FILE <- "results/camera_rank_fixed001_reference_GO_BP_protein_coding.csv"
CAM_001_FDR_FILE <- "results/camera_rank_fixed001_reference_GO_BP_protein_coding_FDR_010.csv"

CAM_COMPARISON_FILE <- "results/camera_GO_BP_protein_coding_primary_stress_comparison.csv"
SUMMARY_FILE <- "results/protein_coding_GO_core_sensitivity_summary.csv"

set.seed(RANDOM_SEED)


3. Load the protein-coding universe and batch-adjusted gene statistics

The protein-coding universe comes from the classification notebook and is intersected with the batch-adjusted differential-expression table.

The selected protein-coding list is checked separately so that the expected 225-gene filtered DE set is documented.


In [3]:
required_files <- c(
    DE_RESULTS_FILE,
    EXPRESSION_FILE,
    SAMPLE_FILE,
    PROTEIN_CODING_UNIVERSE_FILE,
    PROTEIN_CODING_SELECTED_FILE
)

for (f in required_files) {
    if (!file.exists(f)) stop("Missing file: ", f)
}

parse_logical <- function(x) {
    if (is.logical(x)) return(x)

    z <- tolower(trimws(as.character(x)))
    out <- rep(NA, length(z))
    out[z %in% c("true", "t", "1", "yes", "y")] <- TRUE
    out[z %in% c("false", "f", "0", "no", "n")] <- FALSE

    if (any(is.na(out))) {
        stop("Could not parse the selected column.")
    }

    out
}

de_results <- read.csv(
    DE_RESULTS_FILE,
    check.names = FALSE,
    stringsAsFactors = FALSE
)

required_de_columns <- c(
    "Gene",
    "moderated_t_zero",
    "human_minus_chimp_log2_difference",
    "selected"
)

missing_de_columns <- setdiff(required_de_columns, names(de_results))
if (length(missing_de_columns) > 0) {
    stop(
        "Missing DE columns: ",
        paste(missing_de_columns, collapse = ", ")
    )
}

de_results$Gene <- toupper(trimws(as.character(de_results$Gene)))
de_results$moderated_t_zero <- as.numeric(de_results$moderated_t_zero)
de_results$human_minus_chimp_log2_difference <- as.numeric(
    de_results$human_minus_chimp_log2_difference
)
de_results$selected <- parse_logical(de_results$selected)

de_results <- de_results[
    !is.na(de_results$Gene) &
    nzchar(de_results$Gene) &
    is.finite(de_results$moderated_t_zero),
    ,
    drop = FALSE
]

de_results <- de_results[
    !duplicated(de_results$Gene),
    ,
    drop = FALSE
]

pc_all_table <- read.csv(
    PROTEIN_CODING_UNIVERSE_FILE,
    stringsAsFactors = FALSE,
    check.names = FALSE
)

pc_selected_table <- read.csv(
    PROTEIN_CODING_SELECTED_FILE,
    stringsAsFactors = FALSE,
    check.names = FALSE
)

if (!"Gene" %in% names(pc_all_table)) {
    stop("Protein-coding universe file must contain a Gene column.")
}

if (!"Gene" %in% names(pc_selected_table)) {
    stop("Selected protein-coding file must contain a Gene column.")
}

pc_all <- unique(
    toupper(trimws(as.character(pc_all_table$Gene)))
)
pc_all <- pc_all[
    !is.na(pc_all) &
    nzchar(pc_all)
]

pc_selected <- unique(
    toupper(trimws(as.character(pc_selected_table$Gene)))
)
pc_selected <- pc_selected[
    !is.na(pc_selected) &
    nzchar(pc_selected)
]

de_pc <- de_results[
    de_results$Gene %in% pc_all,
    ,
    drop = FALSE
]

selected_pc_in_de <- intersect(
    pc_selected,
    de_pc$Gene
)

cat("All genes in batch-adjusted DE table:", nrow(de_results), "\n")
cat("Protein-coding genes in ranked universe:", nrow(de_pc), "\n")
cat("Protein-coding genes in selected list:", length(pc_selected), "\n")
cat("Selected protein-coding genes found in DE table:", length(selected_pc_in_de), "\n")

cat(
    "Selected protein-coding Human-higher:",
    sum(
        de_pc$selected &
        de_pc$Gene %in% selected_pc_in_de &
        de_pc$human_minus_chimp_log2_difference > 0
    ),
    "\n"
)

cat(
    "Selected protein-coding Chimpanzee-higher:",
    sum(
        de_pc$selected &
        de_pc$Gene %in% selected_pc_in_de &
        de_pc$human_minus_chimp_log2_difference < 0
    ),
    "\n"
)


All genes in batch-adjusted DE table: 11667 
Protein-coding genes in ranked universe: 11052 
Protein-coding genes in selected list: 225 
Selected protein-coding genes found in DE table: 225 
Selected protein-coding Human-higher: 119 
Selected protein-coding Chimpanzee-higher: 106 


4. Ranked directional GOfuncR

All protein-coding genes in the tested universe are ranked by the signed batch-adjusted moderated species t-statistic.

FWER_high_rank corresponds to Human-higher expression.

FWER_low_rank corresponds to Chimpanzee-higher expression.

GOfuncR uses its own annotation database, so its effective annotated universe can differ slightly from the org.Hs.eg.db universe used by Freedman-Lane and CAMERA.


In [4]:
input_ranked <- data.frame(
    gene_id = de_pc$Gene,
    score = de_pc$moderated_t_zero,
    stringsAsFactors = FALSE
)

set.seed(RANDOM_SEED)

ranked_go <- go_enrich(
    genes = input_ranked,
    test = "wilcoxon",
    n_randsets = N_RANDSETS,
    organismDb = "Homo.sapiens",
    domains = "biological_process",
    silent = FALSE
)

ranked_stats <- as.data.frame(
    ranked_go[[1]],
    stringsAsFactors = FALSE
)

required_ranked_columns <- c(
    "node_id",
    "node_name",
    "raw_p_low_rank",
    "raw_p_high_rank",
    "FWER_low_rank",
    "FWER_high_rank"
)

missing_ranked_columns <- setdiff(
    required_ranked_columns,
    names(ranked_stats)
)

if (length(missing_ranked_columns) > 0) {
    stop(
        "Unexpected GOfuncR output. Missing: ",
        paste(missing_ranked_columns, collapse = ", ")
    )
}

ranked_stats$best_FWER <- pmin(
    ranked_stats$FWER_high_rank,
    ranked_stats$FWER_low_rank,
    na.rm = TRUE
)

ranked_stats$direction <- ifelse(
    ranked_stats$FWER_high_rank <= ranked_stats$FWER_low_rank,
    "Human-higher",
    "Chimpanzee-higher"
)

ranked_stats$significant_FWER_0_05 <- (
    ranked_stats$FWER_high_rank <= FWER_CUTOFF |
    ranked_stats$FWER_low_rank <= FWER_CUTOFF
)

ranked_stats <- ranked_stats[
    order(
        ranked_stats$best_FWER,
        pmin(
            ranked_stats$raw_p_high_rank,
            ranked_stats$raw_p_low_rank
        ),
        ranked_stats$node_id
    ),
    ,
    drop = FALSE
]

ranked_significant <- ranked_stats[
    ranked_stats$significant_FWER_0_05,
    ,
    drop = FALSE
]

write.csv(
    ranked_stats,
    GOFUNCR_ALL_FILE,
    row.names = FALSE
)

write.csv(
    ranked_significant,
    GOFUNCR_SIG_FILE,
    row.names = FALSE
)

cat("GOfuncR GO terms tested:", nrow(ranked_stats), "\n")
cat(
    "Human-higher terms at FWER <= 0.05:",
    sum(ranked_stats$FWER_high_rank <= FWER_CUTOFF, na.rm = TRUE),
    "\n"
)
cat(
    "Chimpanzee-higher terms at FWER <= 0.05:",
    sum(ranked_stats$FWER_low_rank <= FWER_CUTOFF, na.rm = TRUE),
    "\n"
)

head(
    ranked_stats[
        ,
        c(
            "node_id",
            "node_name",
            "direction",
            "FWER_high_rank",
            "FWER_low_rank",
            "best_FWER"
        ),
        drop = FALSE
    ],
    30
)


Checking arguments...

Write temporary files...

Find associated categories using database 'Homo.sapiens'...

Remove annotated categories not present in GO-graph...

Found root_nodes: biological_process, cellular_component, molecular_function

Remove invalid genes...

Warning message in go_enrich(genes = input_ranked, test = "wilcoxon", n_randsets = N_RANDSETS, :
“No GO-annotation for genes: IRG1, C2ORF80, KRBA2, WDR78, LINC00634, CCDC171, FAM126A, CENPJ, ATP5D, C9ORF114, PCNXL4, HSPB11, FAM71E1, HRASLS5, FAM149A, C6ORF52, C12ORF42, FAM107B, C12ORF49, C1ORF131, EFCAB2, C11ORF63, C8ORF59, CCDC65, FAM127C, ZBED8, RUNDC3B, MARCH8, N6AMT1, SHFM1, C7ORF31, LINC00282, C5ORF63, DFNA5, XRCC6BP1, APOA1BP, METTL21B, TMEM159, METTL20, HIST1H3I, HIST1H1E, C2ORF40, SOWAHA, FAM200B, BHLHB9, FAM104B, PLA2G16, ANKRD32, C8ORF46, ATP5B, UTP11L, PCP4L1, KIAA1045, RFWD2, FAM122A, CCDC84, C5ORF28, NDUFA4, CCDC176, EFCAB13, SEPW1, DAK, SEPT8, KIAA1462, DIEXF, C17ORF85, HIST1H4A, ICT1, DSCR3, MARCH11, LRRC6,

Read 47205 terms.
Found 28642 nodes.
Graph created.
Data and annotation file parsed.
Number of randomsets: 10000.
Computing randomsets...10000.
Finished

Evaluating randomsets: 
No. of significant ontology nodes for
low ranks				high ranks
of candidate genes at p-value thresholds
0.1	0.05	0.01	0.001	0.0001	0.1	0.05	0.01	0.001	0.0001
1569	652	64	3	0	1400	546	56	3	0	
1456	564	49	0	0	1521	596	71	3	2	
1325	548	71	1	0	1544	631	84	2	0	
1926	859	180	24	5	1199	445	69	5	0	
1760	798	90	7	0	1343	562	84	3	0	
1472	571	79	2	0	1440	575	88	4	0	
1544	622	90	7	0	1329	554	73	0	0	
1543	739	123	17	2	1381	534	69	8	1	
1288	494	54	5	0	1837	860	144	8	0	
1634	747	107	10	3	1284	504	70	4	0	
1423	618	59	1	0	1415	530	58	4	1	
1213	456	56	3	0	1749	827	151	17	3	
1575	638	78	4	0	1408	570	71	4	0	
1419	577	61	3	0	1559	659	105	5	1	
1465	590	83	7	1	1563	687	107	5	0	
1406	539	76	4	0	1457	608	74	0	0	
1345	549	83	3	0	1531	604	77	5	0	
1444	619	88	2	0	1378	566	84	4	0	
1486	664	84	9	3	1546	649	93	5	0	
1985	898	159	20	3	1377	590	


Done.



GOfuncR GO terms tested: 13780 
Human-higher terms at FWER <= 0.05: 88 
Chimpanzee-higher terms at FWER <= 0.05: 185 


,node_id,node_name,direction,FWER_high_rank,FWER_low_rank,best_FWER
,<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>
1,GO:0006139,nucleobase-containing compound metabolic process,Human-higher,0,1,0
2,GO:0006725,cellular aromatic compound metabolic process,Human-higher,0,1,0
3,GO:0006807,nitrogen compound metabolic process,Human-higher,0,1,0
4,GO:0009059,macromolecule biosynthetic process,Human-higher,0,1,0
5,GO:0010467,gene expression,Human-higher,0,1,0
6,GO:0016070,RNA metabolic process,Human-higher,0,1,0
7,GO:0034641,cellular nitrogen compound metabolic process,Human-higher,0,1,0
8,GO:0034645,cellular macromolecule biosynthetic process,Human-higher,0,1,0
9,GO:0043170,macromolecule metabolic process,Human-higher,0,1,0


In [5]:
refined_human <- GOfuncR::refine(
    res = ranked_go,
    fwer = FWER_CUTOFF,
    fwer_col = "FWER_high_rank"
)

refined_chimp <- GOfuncR::refine(
    res = ranked_go,
    fwer = FWER_CUTOFF,
    fwer_col = "FWER_low_rank"
)

human_terms <- refined_human[
    !is.na(refined_human$signif) &
    refined_human$signif,
    ,
    drop = FALSE
]

chimp_terms <- refined_chimp[
    !is.na(refined_chimp$signif) &
    refined_chimp$signif,
    ,
    drop = FALSE
]

write.csv(
    human_terms,
    GOFUNCR_REFINED_HUMAN_FILE,
    row.names = FALSE
)

write.csv(
    chimp_terms,
    GOFUNCR_REFINED_CHIMP_FILE,
    row.names = FALSE
)

cat("Refined Human-higher terms:", nrow(human_terms), "\n")
cat("Refined Chimpanzee-higher terms:", nrow(chimp_terms), "\n")


Found 88 significant categories at FWER threshold 0.05


Get annotations for categories:

find child nodes of GO-categories...



find genes annotated to child nodes using database 'Homo.sapiens'...

Remove annotated categories not present in GO-graph...

Done.


Get annotations for root nodes:

find child nodes of GO-categories...

find genes annotated to child nodes using database 'Homo.sapiens'...

Remove annotated categories not present in GO-graph...

Done.


Refine root node biological_process

using p-value threshold 1.95719774251923e-05...


Found 13 leaves...

13 of the 13 leaves stay significant after refinement.

Refine next level...

Found 14 leaves...

0 of the 14 leaves stay significant after refinement.

Refine next level...

Found 14 leaves...

3 of the 14 leaves stay significant after refinement.

Refine next level...

Found 9 leaves...

2 of the 9 leaves stay significant after refinement.

Refine next level...

Found 8 leaves...

1 of the 8 leaves stay significant after refinement.

Refine next level...

Found 7 leaves...

0 of the 7 leaves stay significant after refinement.

Refine next level..

Refined Human-higher terms: 19 
Refined Chimpanzee-higher terms: 51 


5. Load expression and sample metadata

The expression matrix is restricted to the protein-coding genes that entered the batch-adjusted differential-expression analysis.

The full model contains age, species, and batch.

The reduced Freedman-Lane model contains age and batch but omits species.


In [6]:
expr_all <- read.csv(
    EXPRESSION_FILE,
    row.names = 1,
    check.names = FALSE
)

sample_meta <- read.csv(
    SAMPLE_FILE,
    check.names = FALSE,
    stringsAsFactors = FALSE
)

required_sample_columns <- c(
    "GSM",
    "Species",
    "Age_log2_centered",
    "Batch"
)

missing_sample_columns <- setdiff(
    required_sample_columns,
    names(sample_meta)
)

if (length(missing_sample_columns) > 0) {
    stop(
        "Missing sample columns: ",
        paste(missing_sample_columns, collapse = ", ")
    )
}

sample_meta$GSM <- as.character(sample_meta$GSM)

missing_samples <- setdiff(
    sample_meta$GSM,
    colnames(expr_all)
)

if (length(missing_samples) > 0) {
    stop(
        "Expression matrix is missing samples: ",
        paste(missing_samples, collapse = ", ")
    )
}

expr <- as.matrix(
    expr_all[
        ,
        sample_meta$GSM,
        drop = FALSE
    ]
)

storage.mode(expr) <- "double"
rownames(expr) <- toupper(trimws(rownames(expr)))

if (anyDuplicated(rownames(expr))) {
    stop("Duplicate gene symbols found after uppercasing expression row names.")
}

finite_rows <- apply(
    expr,
    1,
    function(x) all(is.finite(x))
)

expr <- expr[
    finite_rows,
    ,
    drop = FALSE
]

common_pc_genes <- intersect(
    rownames(expr),
    de_pc$Gene
)

expr <- expr[
    common_pc_genes,
    ,
    drop = FALSE
]

sample_meta$Species <- factor(
    sample_meta$Species,
    levels = c("Chimpanzee", "Human")
)

sample_meta$Batch <- factor(
    sample_meta$Batch,
    levels = c("Batch1", "Batch2")
)

design_full <- model.matrix(
    ~ Age_log2_centered + Species + Batch,
    data = sample_meta
)

design_reduced <- model.matrix(
    ~ Age_log2_centered + Batch,
    data = sample_meta
)

if (qr(design_full)$rank != ncol(design_full)) {
    stop("Full design matrix is rank deficient.")
}

if (qr(design_reduced)$rank != ncol(design_reduced)) {
    stop("Reduced design matrix is rank deficient.")
}

species_column <- which(
    colnames(design_full) == "SpeciesHuman"
)

if (length(species_column) != 1L) {
    stop("Could not identify SpeciesHuman in the full design.")
}

cat("Protein-coding genes in expression universe:", nrow(expr), "\n")
cat("Samples:", ncol(expr), "\n")
cat(
    "Full-model columns:",
    paste(colnames(design_full), collapse = ", "),
    "\n"
)


Protein-coding genes in expression universe: 11052 
Samples: 9 
Full-model columns: (Intercept), Age_log2_centered, SpeciesHuman, BatchBatch2 


6. Build the GO Biological Process universe

GOALL is used so that genes annotated to descendant terms also contribute to parent Biological Process terms.

Only GO sets containing 10 to 500 protein-coding genes from the tested expression universe are retained.

This org.Hs.eg.db universe is used by both Freedman-Lane tests and CAMERA.


In [7]:
go_map <- AnnotationDbi::select(
    org.Hs.eg.db,
    keys = rownames(expr),
    columns = c("GOALL", "ONTOLOGYALL"),
    keytype = "SYMBOL"
)

go_map <- unique(
    go_map[
        !is.na(go_map$GOALL) &
        go_map$ONTOLOGYALL == "BP" &
        go_map$SYMBOL %in% rownames(expr),
        c("SYMBOL", "GOALL"),
        drop = FALSE
    ]
)

go_sets <- split(
    go_map$SYMBOL,
    go_map$GOALL
)

go_sets <- lapply(
    go_sets,
    unique
)

go_sets <- go_sets[
    lengths(go_sets) >= MIN_GO_SIZE &
    lengths(go_sets) <= MAX_GO_SIZE
]

go_indices <- limma::ids2indices(
    go_sets,
    identifiers = rownames(expr),
    remove.empty = TRUE
)

go_indices <- go_indices[
    lengths(go_indices) >= MIN_GO_SIZE &
    lengths(go_indices) <= MAX_GO_SIZE
]

go_ids <- names(go_indices)

go_terms <- AnnotationDbi::select(
    GO.db,
    keys = go_ids,
    columns = "TERM",
    keytype = "GOID"
)

go_terms <- go_terms[
    !duplicated(go_terms$GOID),
    ,
    drop = FALSE
]

go_names <- go_terms$TERM[
    match(
        go_ids,
        go_terms$GOID
    )
]

cat(
    "Eligible protein-coding GO Biological Process sets:",
    length(go_indices),
    "\n"
)


'select()' returned 1:many mapping between keys and columns

'select()' returned 1:1 mapping between keys and columns



Eligible protein-coding GO Biological Process sets: 4980 


7. Moderated species t-statistic for Freedman-Lane

For each observed or permuted expression matrix, the full age + species + batch model is fitted gene by gene.

Residual variances are moderated across genes with limma::squeezeVar before the species t-statistics are calculated.

Both Freedman-Lane statistics therefore use moderated rather than ordinary OLS t-statistics.


In [8]:
Y <- t(expr)
X_full <- design_full
X_reduced <- design_reduced

n_samples <- nrow(X_full)
n_genes <- ncol(Y)
df_full <- n_samples - ncol(X_full)

if (df_full <= 0) {
    stop("No residual degrees of freedom in the full model.")
}

beta_reduced <- solve(
    crossprod(X_reduced),
    crossprod(X_reduced, Y)
)

fitted_reduced <- X_reduced %*% beta_reduced
residuals_reduced <- Y - fitted_reduced

XtX_inv <- solve(
    crossprod(X_full)
)

moderated_species_t <- function(Y_matrix) {

    beta <- XtX_inv %*%
        crossprod(
            X_full,
            Y_matrix
        )

    residuals <- Y_matrix -
        X_full %*% beta

    sigma2 <- colSums(
        residuals^2
    ) / df_full

    sv <- limma::squeezeVar(
        sigma2,
        df = df_full
    )

    se_species <- sqrt(
        sv$var.post *
        XtX_inv[
            species_column,
            species_column
        ]
    )

    as.numeric(
        beta[
            species_column,
            ,
            drop = TRUE
        ] / se_species
    )
}

t_observed <- moderated_species_t(Y)

cat(
    "Observed moderated species t-statistics calculated for",
    length(t_observed),
    "genes.\n"
)


Observed moderated species t-statistics calculated for 11052 genes.


8. Mean-t and mean-rank Freedman-Lane tests

Both statistics use the same 5,000 residual permutations.

Mean-t is the mean moderated species t-statistic within each GO set.

Mean-rank ranks the moderated species t-statistics genome-wide and uses the centered mean rank within each GO set.

The same sample permutation is applied to every gene in a given permutation, preserving the observed gene-gene dependence structure.

Both analyses report two-sided empirical p-values, BH FDR, and max-statistic FWER.


In [9]:
set_sizes <- lengths(go_indices)

membership <- Matrix::sparseMatrix(
    i = rep(
        seq_along(go_indices),
        times = set_sizes
    ),
    j = unlist(
        go_indices,
        use.names = FALSE
    ),
    x = rep(
        1 / set_sizes,
        times = set_sizes
    ),
    dims = c(
        length(go_indices),
        nrow(expr)
    )
)

mean_t_stat <- function(t_mat) {

    if (is.null(dim(t_mat))) {
        t_mat <- matrix(
            t_mat,
            ncol = 1L
        )
    }

    as.matrix(
        membership %*% t_mat
    )
}

rank_stat <- function(t_mat) {

    if (is.null(dim(t_mat))) {
        t_mat <- matrix(
            t_mat,
            ncol = 1L
        )
    }

    r <- apply(
        t_mat,
        2,
        rank,
        ties.method = "average"
    )

    if (is.null(dim(r))) {
        r <- matrix(
            r,
            ncol = 1L
        )
    }

    as.matrix(
        membership %*% r
    ) - (
        nrow(t_mat) + 1
    ) / 2
}

observed_meant <- mean_t_stat(
    matrix(
        t_observed,
        ncol = 1L
    )
)[, 1]

observed_rank <- rank_stat(
    matrix(
        t_observed,
        ncol = 1L
    )
)[, 1]

meant_exceed <- integer(
    length(go_indices)
)

rank_exceed <- integer(
    length(go_indices)
)

meant_max_null <- numeric(
    N_PERMUTATIONS
)

rank_max_null <- numeric(
    N_PERMUTATIONS
)

set.seed(RANDOM_SEED)
perm_number <- 0L

while (perm_number < N_PERMUTATIONS) {

    block_n <- min(
        PERM_BLOCK,
        N_PERMUTATIONS - perm_number
    )

    t_block <- matrix(
        NA_real_,
        nrow = n_genes,
        ncol = block_n
    )

    for (b in seq_len(block_n)) {

        perm <- sample.int(
            n_samples
        )

        Y_perm <- fitted_reduced +
            residuals_reduced[
                perm,
                ,
                drop = FALSE
            ]

        t_block[, b] <- moderated_species_t(
            Y_perm
        )
    }

    meant_block <- mean_t_stat(
        t_block
    )

    rank_block <- rank_stat(
        t_block
    )

    meant_exceed <- meant_exceed +
        rowSums(
            abs(meant_block) >=
            abs(observed_meant)
        )

    rank_exceed <- rank_exceed +
        rowSums(
            abs(rank_block) >=
            abs(observed_rank)
        )

    idx <- (
        perm_number + 1L
    ):(
        perm_number + block_n
    )

    meant_max_null[idx] <- apply(
        abs(meant_block),
        2,
        max,
        na.rm = TRUE
    )

    rank_max_null[idx] <- apply(
        abs(rank_block),
        2,
        max,
        na.rm = TRUE
    )

    perm_number <- perm_number + block_n

    if (
        perm_number %% 500L == 0L ||
        perm_number == N_PERMUTATIONS
    ) {
        cat(
            "Freedman-Lane permutations:",
            perm_number,
            "of",
            N_PERMUTATIONS,
            "\n"
        )
    }
}

empirical_p <- function(exceed) {
    (
        exceed + 1
    ) / (
        N_PERMUTATIONS + 1
    )
}

maxstat_fwer <- function(observed, max_null) {
    vapply(
        abs(observed),
        function(obs) {
            (
                1 +
                sum(
                    max_null >= obs,
                    na.rm = TRUE
                )
            ) / (
                N_PERMUTATIONS + 1
            )
        },
        numeric(1)
    )
}

meant_p <- empirical_p(
    meant_exceed
)

rank_p <- empirical_p(
    rank_exceed
)

meant_fdr <- p.adjust(
    meant_p,
    method = "BH"
)

rank_fdr <- p.adjust(
    rank_p,
    method = "BH"
)

meant_fwer <- maxstat_fwer(
    observed_meant,
    meant_max_null
)

rank_fwer <- maxstat_fwer(
    observed_rank,
    rank_max_null
)

meant_fl_result <- data.frame(
    node_id = go_ids,
    node_name = go_names,
    NGenes = set_sizes,
    mean_moderated_t = observed_meant,
    species_direction = ifelse(
        observed_meant > 0,
        "Human-higher",
        ifelse(
            observed_meant < 0,
            "Chimpanzee-higher",
            "Centered"
        )
    ),
    permutation_p = meant_p,
    FDR_BH = meant_fdr,
    FWER_maxstat = meant_fwer,
    stringsAsFactors = FALSE
)

rank_fl_result <- data.frame(
    node_id = go_ids,
    node_name = go_names,
    NGenes = set_sizes,
    centered_mean_rank = observed_rank,
    species_direction = ifelse(
        observed_rank > 0,
        "Human-higher",
        ifelse(
            observed_rank < 0,
            "Chimpanzee-higher",
            "Centered"
        )
    ),
    permutation_p = rank_p,
    FDR_BH = rank_fdr,
    FWER_maxstat = rank_fwer,
    stringsAsFactors = FALSE
)

meant_fl_result <- meant_fl_result[
    order(
        meant_fl_result$FWER_maxstat,
        meant_fl_result$FDR_BH,
        meant_fl_result$permutation_p
    ),
    ,
    drop = FALSE
]

rank_fl_result <- rank_fl_result[
    order(
        rank_fl_result$FDR_BH,
        rank_fl_result$FWER_maxstat,
        rank_fl_result$permutation_p
    ),
    ,
    drop = FALSE
]

meant_fl_fdr <- meant_fl_result[
    !is.na(meant_fl_result$FDR_BH) &
    meant_fl_result$FDR_BH <= FDR_CUTOFF,
    ,
    drop = FALSE
]

meant_fl_fwer <- meant_fl_result[
    !is.na(meant_fl_result$FWER_maxstat) &
    meant_fl_result$FWER_maxstat <= FWER_CUTOFF,
    ,
    drop = FALSE
]

rank_fl_fdr <- rank_fl_result[
    !is.na(rank_fl_result$FDR_BH) &
    rank_fl_result$FDR_BH <= FDR_CUTOFF,
    ,
    drop = FALSE
]

rank_fl_fwer <- rank_fl_result[
    !is.na(rank_fl_result$FWER_maxstat) &
    rank_fl_result$FWER_maxstat <= FWER_CUTOFF,
    ,
    drop = FALSE
]

write.csv(
    meant_fl_result,
    FL_T_ALL_FILE,
    row.names = FALSE
)

write.csv(
    meant_fl_fdr,
    FL_T_FDR_FILE,
    row.names = FALSE
)

write.csv(
    meant_fl_fwer,
    FL_T_FWER_FILE,
    row.names = FALSE
)

write.csv(
    rank_fl_result,
    FL_RANK_ALL_FILE,
    row.names = FALSE
)

write.csv(
    rank_fl_fdr,
    FL_RANK_FDR_FILE,
    row.names = FALSE
)

write.csv(
    rank_fl_fwer,
    FL_RANK_FWER_FILE,
    row.names = FALSE
)

cat("\nMean-t Freedman-Lane\n")
cat("Terms at BH-FDR <= 0.10:", nrow(meant_fl_fdr), "\n")
cat("Terms at max-stat FWER <= 0.05:", nrow(meant_fl_fwer), "\n")

cat("\nMean-rank Freedman-Lane\n")
cat("Terms at BH-FDR <= 0.10:", nrow(rank_fl_fdr), "\n")
cat("Terms at max-stat FWER <= 0.05:", nrow(rank_fl_fwer), "\n")

head(meant_fl_result, 30)


Freedman-Lane permutations: 500 of 10000 
Freedman-Lane permutations: 1000 of 10000 
Freedman-Lane permutations: 1500 of 10000 
Freedman-Lane permutations: 2000 of 10000 
Freedman-Lane permutations: 2500 of 10000 
Freedman-Lane permutations: 3000 of 10000 
Freedman-Lane permutations: 3500 of 10000 
Freedman-Lane permutations: 4000 of 10000 
Freedman-Lane permutations: 4500 of 10000 
Freedman-Lane permutations: 5000 of 10000 
Freedman-Lane permutations: 5500 of 10000 
Freedman-Lane permutations: 6000 of 10000 
Freedman-Lane permutations: 6500 of 10000 
Freedman-Lane permutations: 7000 of 10000 
Freedman-Lane permutations: 7500 of 10000 
Freedman-Lane permutations: 8000 of 10000 
Freedman-Lane permutations: 8500 of 10000 
Freedman-Lane permutations: 9000 of 10000 
Freedman-Lane permutations: 9500 of 10000 
Freedman-Lane permutations: 10000 of 10000 

Mean-t Freedman-Lane
Terms at BH-FDR <= 0.10: 0 
Terms at max-stat FWER <= 0.05: 4 

Mean-rank Freedman-Lane
Terms at BH-FDR <= 0.10: 0 
Te

,node_id,node_name,NGenes,mean_moderated_t,species_direction,permutation_p,FDR_BH,FWER_maxstat
,<chr>,<chr>,<int>,<dbl>,<chr>,<dbl>,<dbl>,<dbl>
GO:0035437,GO:0035437,maintenance of protein localization in endoplasmic reticulum,11,2.512517,Human-higher,0.00099990,0.1998519,0.02349765
GO:0090266,GO:0090266,regulation of mitotic cell cycle spindle assembly checkpoint,13,2.353812,Human-higher,0.00509949,0.1998519,0.04269573
GO:1903504,GO:1903504,regulation of mitotic spindle checkpoint,13,2.353812,Human-higher,0.00509949,0.1998519,0.04269573
GO:0016446,GO:0016446,somatic hypermutation of immunoglobulin genes,10,2.318211,Human-higher,0.00649935,0.1998519,0.04679532
GO:0031639,GO:0031639,plasminogen activation,10,-2.139939,Chimpanzee-higher,0.02009799,0.1998519,0.07809219
GO:1990709,GO:1990709,presynaptic active zone organization,11,-2.012255,Chimpanzee-higher,0.02979702,0.2079207,0.10748925
GO:0034501,GO:0034501,protein localization to kinetochore,11,1.973807,Human-higher,0.00099990,0.1998519,0.11698830
GO:1903083,GO:1903083,protein localization to condensed chromosome,11,1.973807,Human-higher,0.00099990,0.1998519,0.11698830
GO:0045762,GO:0045762,positive regulation of adenylate cyclase activity,11,-1.960914,Chimpanzee-higher,0.01129887,0.1998519,0.12048795


9. Rank-based CAMERA

The primary CAMERA specification estimates inter-gene correlation separately for each GO set.

The fixed-correlation 0.05 run is a stress test that assumes stronger positive within-set correlation.

The fixed-correlation 0.01 run is optional reference only and is disabled by default.

All CAMERA runs use ranks and trend.var = TRUE.


In [10]:
run_camera <- function(inter_gene_cor) {

    res <- limma::camera(
        y = expr,
        index = go_indices,
        design = design_full,
        contrast = species_column,
        use.ranks = TRUE,
        inter.gene.cor = inter_gene_cor,
        trend.var = TRUE,
        sort = FALSE
    )

    res$node_id <- rownames(res)
    rownames(res) <- NULL

    res$node_name <- go_names[
        match(
            res$node_id,
            go_ids
        )
    ]

    res$species_direction <- ifelse(
        res$Direction == "Up",
        "Human-higher",
        "Chimpanzee-higher"
    )

    res[
        order(
            res$FDR,
            res$PValue
        ),
        ,
        drop = FALSE
    ]
}

camera_est <- run_camera(
    NA
)

camera_005 <- run_camera(
    0.05
)

camera_est_fdr <- camera_est[
    !is.na(camera_est$FDR) &
    camera_est$FDR <= FDR_CUTOFF,
    ,
    drop = FALSE
]

camera_005_fdr <- camera_005[
    !is.na(camera_005$FDR) &
    camera_005$FDR <= FDR_CUTOFF,
    ,
    drop = FALSE
]

write.csv(
    camera_est,
    CAM_EST_ALL_FILE,
    row.names = FALSE
)

write.csv(
    camera_est_fdr,
    CAM_EST_FDR_FILE,
    row.names = FALSE
)

write.csv(
    camera_005,
    CAM_005_ALL_FILE,
    row.names = FALSE
)

write.csv(
    camera_005_fdr,
    CAM_005_FDR_FILE,
    row.names = FALSE
)

camera_001 <- NULL
camera_001_fdr <- NULL

if (RUN_CAMERA_001_REFERENCE) {

    camera_001 <- run_camera(
        0.01
    )

    camera_001_fdr <- camera_001[
        !is.na(camera_001$FDR) &
        camera_001$FDR <= FDR_CUTOFF,
        ,
        drop = FALSE
    ]

    write.csv(
        camera_001,
        CAM_001_ALL_FILE,
        row.names = FALSE
    )

    write.csv(
        camera_001_fdr,
        CAM_001_FDR_FILE,
        row.names = FALSE
    )
}

cat(
    "CAMERA estimated-correlation terms at FDR <= 0.10:",
    nrow(camera_est_fdr),
    "\n"
)

cat(
    "CAMERA fixed 0.05 terms at FDR <= 0.10:",
    nrow(camera_005_fdr),
    "\n"
)

if (RUN_CAMERA_001_REFERENCE) {
    cat(
        "CAMERA fixed 0.01 reference terms at FDR <= 0.10:",
        nrow(camera_001_fdr),
        "\n"
    )
}


CAMERA estimated-correlation terms at FDR <= 0.10: 0 
CAMERA fixed 0.05 terms at FDR <= 0.10: 0 
CAMERA fixed 0.01 reference terms at FDR <= 0.10: 214 


10. CAMERA comparison table

The primary estimated-correlation result and fixed-0.05 stress test are placed side by side by GO identifier.

If the optional fixed-0.01 reference run is enabled, its p-values, FDR values, and directions are appended to the same comparison table.


In [11]:
camera_keep <- function(x, prefix) {

    out <- x[
        ,
        c(
            "node_id",
            "node_name",
            "NGenes",
            "species_direction",
            "PValue",
            "FDR"
        ),
        drop = FALSE
    ]

    names(out)[
        names(out) == "species_direction"
    ] <- paste0(
        prefix,
        "_direction"
    )

    names(out)[
        names(out) == "PValue"
    ] <- paste0(
        prefix,
        "_PValue"
    )

    names(out)[
        names(out) == "FDR"
    ] <- paste0(
        prefix,
        "_FDR"
    )

    out
}

cam_cmp <- camera_keep(
    camera_est,
    "estimated"
)

cam_cmp_005 <- camera_keep(
    camera_005,
    "fixed005"
)[
    ,
    c(
        "node_id",
        "fixed005_direction",
        "fixed005_PValue",
        "fixed005_FDR"
    ),
    drop = FALSE
]

cam_cmp <- merge(
    cam_cmp,
    cam_cmp_005,
    by = "node_id",
    all = TRUE,
    sort = FALSE
)

if (RUN_CAMERA_001_REFERENCE) {

    cam_cmp_001 <- camera_keep(
        camera_001,
        "fixed001_reference"
    )[
        ,
        c(
            "node_id",
            "fixed001_reference_direction",
            "fixed001_reference_PValue",
            "fixed001_reference_FDR"
        ),
        drop = FALSE
    ]

    cam_cmp <- merge(
        cam_cmp,
        cam_cmp_001,
        by = "node_id",
        all = TRUE,
        sort = FALSE
    )
}

cam_cmp <- cam_cmp[
    order(
        cam_cmp$estimated_FDR,
        cam_cmp$fixed005_FDR
    ),
    ,
    drop = FALSE
]

write.csv(
    cam_cmp,
    CAM_COMPARISON_FILE,
    row.names = FALSE
)

head(cam_cmp, 40)


,node_id,node_name,NGenes,estimated_direction,estimated_PValue,estimated_FDR,fixed005_direction,fixed005_PValue,fixed005_FDR,fixed001_reference_direction,fixed001_reference_PValue,fixed001_reference_FDR
,<chr>,<chr>,<dbl>,<chr>,<dbl>,<dbl>,<chr>,<dbl>,<dbl>,<chr>,<dbl>,<dbl>
1,GO:0090266,regulation of mitotic cell cycle spindle assembly checkpoint,13,Human-higher,0.009387399,0.9617401,Human-higher,0.0010872151,0.9997546,Human-higher,1.042820e-04,0.018547296
2,GO:1903504,regulation of mitotic spindle checkpoint,13,Human-higher,0.009387399,0.9617401,Human-higher,0.0010872151,0.9997546,Human-higher,1.042820e-04,0.018547296
3,GO:0007052,mitotic spindle organization,91,Human-higher,0.009543979,0.9617401,Human-higher,0.0610888007,0.9997546,Human-higher,1.596174e-03,0.079012587
4,GO:1902850,microtubule cytoskeleton organization involved in mitosis,116,Human-higher,0.012010873,0.9617401,Human-higher,0.1140487472,0.9997546,Human-higher,5.523726e-03,0.107197912
5,GO:0090307,mitotic spindle assembly,51,Human-higher,0.012658880,0.9617401,Human-higher,0.0197135793,0.9997546,Human-higher,4.193515e-04,0.041767408
6,GO:0007051,spindle organization,138,Human-higher,0.013052843,0.9617401,Human-higher,0.1146571898,0.9997546,Human-higher,4.424069e-03,0.100602124
7,GO:0016446,somatic hypermutation of immunoglobulin genes,10,Human-higher,0.016392758,0.9617401,Human-higher,0.0029868254,0.9997546,Human-higher,6.580866e-04,0.052495158
8,GO:0048285,organelle fission,296,Human-higher,0.021869724,0.9617401,Human-higher,0.2899745999,0.9997546,Human-higher,3.589733e-02,0.291949606
9,GO:0030167,proteoglycan catabolic process,12,Chimpanzee-higher,0.022436854,0.9617401,Chimpanzee-higher,0.0082794618,0.9997546,Chimpanzee-higher,1.924152e-03,0.080061468


11. Summary

The summary table reports the number and direction of discoveries for all requested specifications.

The main comparison is:

GOfuncR ranked reference;
mean-t Freedman-Lane with moderated t;
mean-rank Freedman-Lane with moderated t;
CAMERA with estimated correlation;
CAMERA with fixed correlation 0.05.

The fixed-0.01 CAMERA result is added only when the optional reference flag is enabled.


In [12]:
count_direction <- function(df, column = "species_direction") {

    if (is.null(df) || nrow(df) == 0) {
        return(
            c(
                total = 0,
                Human_higher = 0,
                Chimpanzee_higher = 0
            )
        )
    }

    c(
        total = nrow(df),
        Human_higher = sum(
            df[[column]] == "Human-higher",
            na.rm = TRUE
        ),
        Chimpanzee_higher = sum(
            df[[column]] == "Chimpanzee-higher",
            na.rm = TRUE
        )
    )
}

gofuncr_counts <- c(
    total = nrow(ranked_significant),
    Human_higher = sum(
        ranked_significant$direction == "Human-higher",
        na.rm = TRUE
    ),
    Chimpanzee_higher = sum(
        ranked_significant$direction == "Chimpanzee-higher",
        na.rm = TRUE
    )
)

summary_rows <- list(
    gofuncr_counts,
    count_direction(meant_fl_fdr),
    count_direction(meant_fl_fwer),
    count_direction(rank_fl_fdr),
    count_direction(rank_fl_fwer),
    count_direction(camera_est_fdr),
    count_direction(camera_005_fdr)
)

analysis_names <- c(
    "GOfuncR ranked: FWER <= 0.05",
    "Mean-t Freedman-Lane moderated t: BH FDR <= 0.10",
    "Mean-t Freedman-Lane moderated t: max-stat FWER <= 0.05",
    "Mean-rank Freedman-Lane moderated t: BH FDR <= 0.10",
    "Mean-rank Freedman-Lane moderated t: max-stat FWER <= 0.05",
    "CAMERA ranks, estimated correlation: BH FDR <= 0.10",
    "CAMERA ranks, fixed correlation 0.05: BH FDR <= 0.10"
)

if (RUN_CAMERA_001_REFERENCE) {
    summary_rows <- append(
        summary_rows,
        list(
            count_direction(
                camera_001_fdr
            )
        )
    )

    analysis_names <- c(
        analysis_names,
        "CAMERA ranks, fixed correlation 0.01 reference: BH FDR <= 0.10"
    )
}

summary_matrix <- do.call(
    rbind,
    summary_rows
)

summary_table <- data.frame(
    analysis = analysis_names,
    total = as.integer(
        summary_matrix[
            ,
            "total"
        ]
    ),
    Human_higher = as.integer(
        summary_matrix[
            ,
            "Human_higher"
        ]
    ),
    Chimpanzee_higher = as.integer(
        summary_matrix[
            ,
            "Chimpanzee_higher"
        ]
    ),
    stringsAsFactors = FALSE
)

write.csv(
    summary_table,
    SUMMARY_FILE,
    row.names = FALSE
)

summary_table


analysis,total,Human_higher,Chimpanzee_higher
<chr>,<int>,<int>,<int>
GOfuncR ranked: FWER <= 0.05,273,88,185
Mean-t Freedman-Lane moderated t: BH FDR <= 0.10,0,0,0
Mean-t Freedman-Lane moderated t: max-stat FWER <= 0.05,4,4,0
Mean-rank Freedman-Lane moderated t: BH FDR <= 0.10,0,0,0
Mean-rank Freedman-Lane moderated t: max-stat FWER <= 0.05,0,0,0
"CAMERA ranks, estimated correlation: BH FDR <= 0.10",0,0,0
"CAMERA ranks, fixed correlation 0.05: BH FDR <= 0.10",0,0,0
"CAMERA ranks, fixed correlation 0.01 reference: BH FDR <= 0.10",214,52,162


12. Interpretation note

GOfuncR remains the ranked reference analysis, but its random-set null operates at the gene level and therefore does not preserve sample-level gene-gene correlation.

The two Freedman-Lane tests use the same sample-level residual permutations and the same moderated species t-statistics. Mean-t is the main sensitivity analysis because the corresponding full-universe test supported the checkpoint/mitotic module. Mean-rank provides the stricter rank-based comparison and was negative after multiple-testing correction in the full-universe analysis.

For CAMERA, estimated inter-gene correlation is the primary result. Fixed correlation 0.05 is a stress test. Fixed correlation 0.01 is reference-only because the full-universe analysis showed that it was substantially more permissive.

GOfuncR and the other analyses use slightly different annotation machinery, so their exact effective gene and GO universes can differ slightly even after applying the same protein-coding restriction.
